# Tuning a random forest on the heart disease data
A random forest out of the box against three other classifiers, then tuning with `GridSearchCV` and
`RandomizedSearchCV`.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## 1. The data: 303 patients, 13 inputs, target 1 = heart disease

In [ ]:
df = pd.read_csv("data/heart.csv")
print(df.shape)
df.head()

In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 2. Four classifiers, out of the box: one test split, then 10-fold cross-validation

In [ ]:
models = {
    "random forest": RandomForestClassifier(random_state=42),
    "gradient boosting": GradientBoostingClassifier(random_state=42),
    "SVM (SVC)": SVC(),
    "logistic regression": LogisticRegression(max_iter=5000),   # more iterations, so it converges
}
for name, model in models.items():
    test_acc = model.fit(X_train, y_train).score(X_test, y_test)
    cv_acc = cross_val_score(model, X, y, cv=10).mean()          # 10 splits, averaged
    print(f"{name:20s} test split {test_acc:.3f}   10-fold CV {cv_acc:.3f}")

In [ ]:
# SVC measures distances, so it needs scaled inputs: with a scaler in a pipeline it catches up
svc_scaled = make_pipeline(StandardScaler(), SVC())
print("scaled SVC: test split", round(svc_scaled.fit(X_train, y_train).score(X_test, y_test), 3),
      " 10-fold CV", round(cross_val_score(svc_scaled, X, y, cv=10).mean(), 3))

## 3. One hand-tuned setting: 75% of the rows per tree

In [ ]:
rf = RandomForestClassifier(max_samples=0.75, random_state=42)
print("test split:", round(rf.fit(X_train, y_train).score(X_test, y_test), 3))
print("10-fold CV:", round(cross_val_score(rf, X, y, cv=10).mean(), 3))

## 4. GridSearchCV: every combination of 4 hyperparameters

In [ ]:
param_grid = {
    "n_estimators": [20, 60, 100, 120],   # number of trees
    "max_features": [0.2, 0.6, 1.0],      # share of columns considered at each split
    "max_depth": [2, 8, None],            # depth of each tree (None: fully grown)
    "max_samples": [0.5, 0.75, 1.0],      # share of rows per tree
}
print("combinations:", np.prod([len(v) for v in param_grid.values()]))

In [ ]:
start = time.time()
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, verbose=1, n_jobs=-1)
rf_grid.fit(X_train, y_train)
print(f"{time.time() - start:.1f} seconds")
print(rf_grid.best_params_)
print("best cross-validated accuracy:", round(rf_grid.best_score_, 3))
print("test accuracy of the best forest:", round(rf_grid.score(X_test, y_test), 3))

In [ ]:
# the five best combinations
res = pd.DataFrame(rf_grid.cv_results_)
cols = ["param_n_estimators", "param_max_features", "param_max_depth", "param_max_samples", "mean_test_score"]
res.sort_values("rank_test_score")[cols].head()

## 5. RandomizedSearchCV: a bigger grid, 10 random combinations
A first attempt mixes `bootstrap=False` with `max_samples`, which scikit-learn refuses: those candidates score NaN.

In [ ]:
big_grid = dict(param_grid, bootstrap=[True, False], min_samples_split=[2, 5], min_samples_leaf=[1, 2])
print("combinations:", np.prod([len(v) for v in big_grid.values()]))
naive = RandomizedSearchCV(RandomForestClassifier(random_state=42), big_grid, cv=5, n_jobs=-1, random_state=42,
                           error_score=np.nan)
naive.fit(X_train, y_train)
print("candidates that failed:", int(np.isnan(naive.cv_results_["mean_test_score"]).sum()), "of 10")

The fix: a list of two grids. `max_samples` appears only in the grid with `bootstrap=True`.

In [ ]:
grids = [
    dict(param_grid, bootstrap=[True], min_samples_split=[2, 5], min_samples_leaf=[1, 2]),
    {"n_estimators": [20, 60, 100, 120], "max_features": [0.2, 0.6, 1.0], "max_depth": [2, 8, None],
     "bootstrap": [False], "min_samples_split": [2, 5], "min_samples_leaf": [1, 2]},
]
start = time.time()
rf_random = RandomizedSearchCV(RandomForestClassifier(random_state=42), grids, n_iter=10, cv=5, verbose=1,
                               n_jobs=-1, random_state=42)
rf_random.fit(X_train, y_train)
print(f"{time.time() - start:.1f} seconds")
print(rf_random.best_params_)
print("best cross-validated accuracy:", round(rf_random.best_score_, 3))
print("test accuracy of the best forest:", round(rf_random.score(X_test, y_test), 3))